# Tsetlin bake-off on a free Colab GPU

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
Stay on the tab. Run is ~6–8 min.

Colab is on Python 3.13; `tmu` only ships wheels up to 3.12, so cell 3 spins
up a throwaway Python 3.11 (via `uv`) just for the Tsetlin step. Everything
runs as a subprocess so nothing touches the notebook kernel.

In [ ]:
# 1. GPU check + get the code + baseline deps (system Python 3.13)
!nvidia-smi -L || echo 'NO GPU — Runtime > Change runtime type > T4 GPU'
import os
if not os.path.isdir('tsetlin-market-lab'):
    !git clone --depth 1 https://github.com/naibwedi/tsetlin-market-lab.git
os.chdir('/content/tsetlin-market-lab')
!pip -q install pandas pyarrow pyyaml python-dotenv scikit-learn xgboost lightgbm
print('cwd', os.getcwd())

In [ ]:
# 2. Features (synthetic until real odds land) + the 7 baselines
import glob
if not glob.glob('data/features/X.parquet'):
    !python -m src.ingest.make_synthetic --n-matches 80
    !python -m src.panel.build_panel --config config/features.yaml
    !python -m src.features.booleanize --config config/features.yaml
!python -m src.models.bakeoff --config config/bakeoff.ci.yaml
print('\n' + open('results/summary.md').read())

In [ ]:
# 3. Tsetlin Machine on the GPU, inside a Python 3.11 venv (tmu has no 3.13 wheel)
!pip -q install uv
!uv venv /content/tm311 --python 3.11 --quiet
!uv pip install -q --python /content/tm311/bin/python \
    'numpy<2' 'scikit-learn==1.5.2' pandas pyarrow pyyaml python-dotenv tmu pycuda
!cd /content/tsetlin-market-lab && /content/tm311/bin/python -m scripts.tm_run

In [ ]:
# 4. The result + the rules it learned
print(open('results/tm_result.json').read())
print('\n--- clauses ---')
print(open('results/tm_clauses.txt').read())